In [1]:
from google.cloud import bigquery  # Conector oficial do banco de dados analítico (BigQuery) da Google.
from google.oauth2 import service_account  # Mecanismo de autenticação por chave/arquivo JSON de conta de serviço.

# Manipulação e Análise de Dados
import pandas as pd 
import pandas_gbq  

# 3. CONFIGURAÇÃO DE FONTES DE DADOS E INFRAESTRUTURA DE NUVEM
# ==============================================================================

# --- FONTE 1: REPOSITÓRIO REMOTO (GITHUB) ---
# URL base onde estão armazenados arquivos CSV públicos contendo dados históricos de filmes.
url_github_movies_dataset = "https://media.githubusercontent.com/media/lucas-aulas/dataset-movies/refs/heads/main"

# --- FONTE 2: AMBIENTE DE LEITURA (PROJETO COMPARTILHADO DO PROFESSOR) ---
# ID do projeto na Google Cloud de onde leremos uma tabela institucional de bilheterias.
fonte_gcp_project_id = "meu-projeto-eng-dados"

# --- DESTINO: AMBIENTE DE ESCRITA (PROJETO PRIVADO DO ALUNO) ---
# Caminho local do arquivo JSON que contém as credenciais de segurança (chave privada).
# ATENÇÃO: Nunca envie esta chave para o GitHub público! Ela equivale à senha do seu banco de dados.
meu_service_account_file = r"C:\Users\1597779\data_project-template\secrets\meu-projeto-engdados-e9e71c9de509.json"
meu_gcp_project_id = "meu-projeto-engdados"  # ID do seu projeto pessoal criado no GCP.

# Inicialização da autenticação e do cliente BigQuery
# Transforma o arquivo JSON em um objeto de credenciais aceito pelo Google.
gcp_credentials = service_account.Credentials.from_service_account_file(meu_service_account_file)

# O objeto 'client' funciona como a nossa ponte ativa de conexão/comunicação com a nuvem do Google.
bigquery_client = bigquery.Client(credentials=gcp_credentials)

In [2]:
# ==============================================================================
# 4. EXTRAÇÃO DE DADOS (EXTRACT)
# ==============================================================================

# ------------------------------------------------------------------------------
# Fonte: Camada Bronze - BigQuery
# Objetivo:
#   Carregar dados financeiros e comerciais de filmes da Marvel e DC.
#
# Dataset:
#   bronze.boxoffice_dc_marvel
# ------------------------------------------------------------------------------

df_boxoffice_dc_marvel = bigquery_client.query("""
    SELECT *
    FROM bronze.boxoffice_dc_marvel
    """).to_dataframe(create_bqstorage_client=False)

In [3]:
df_boxoffice_dc_marvel.head(5)  

,int64_field_0,FILM,US_RELEASE_DATE,BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BOX_OFFICE_GROSS_WORLDWIDE,BUDGET,MCU,PHASE,DISTRIBUTOR,...,CHARACTER_FAMILY,DOMESTIC,GROSS_TO_BUDGET,ROTTEN_TOMATOES_CRITIC_SCORE,MALE_FEMALE_LED,YEAR,INFLATION_ADJUSTED_WORLDWIDE_GROSS,INFLATION_ADJUSTED_BUDGET,25X_PROD,BREAK_EVEN
0,113,NaN,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN
1,89,The New Mutants,2020-08-28,23852659,25316935,49169594,67000000,False,NaN,20th Century Studios,...,X-Men,0.49,0.73,36.0,Co-starring,2020.0,49169594,67000000,167500000,Flop
2,98,Morbius,2022-04-01,73865530,93595431,167460961,75000000,False,NaN,Columbia Pictures,...,Spider-Man Allies and Villains,0.44,2.23,15.0,Male,2022.0,167460961,75000000,187500000,Flop
3,112,Madame Web,2024-02-14,42619699,54000000,96619699,80000000,False,NaN,Columbia Pictures,...,Spider-Man Allies and Villains,0.44,1.21,13.0,Female,2024.0,96619699,80000000,200000000,Flop
4,21,The Punisher,2004-04-16,33810189,20889916,54700105,33000000,False,NaN,Lions Gate Films,...,Punisher,0.62,1.66,29.0,Male,2004.0,54700105,33000000,82500000,Flop


In [4]:
df_imdb_filmes = bigquery_client.query("""
    SELECT *
    FROM bronze.imdb_filmes
    """).to_dataframe(create_bqstorage_client=False)


In [5]:
df_imdb_filmes.head()

,Unnamed: 0,ID,TITLE,LINK,YEAR,DURATION,RATING_MPA,RATING_IMDB,VOTE,BUDGET,...,WRITER,STAR,GENRE,COUNTRY_ORIGIN,FILMING_LOCATION,PRODUCTION_COMPANY,LANGUAGE,WIN,NOMINATION,OSCAR
0,0,tt0073470,Strip Nude for Your Killer,https://www.imdb.com/title/tt0073470,1975,1h 38m,Not Rated,5.6,33000.0,NaN,...,"Andrea Bianchi, Massimo Felisatti","Edwige Fenech, Nino Castelnuovo, Femi Benussi","Erotic Thriller, Giallo, Slasher Horror, Horro...",Italy,"Milan, Lombardia, Italy",FRAL Cinematografica,Italian,0,0,0
1,1,tt0072764,Carry on Behind,https://www.imdb.com/title/tt0072764,1975,1h 30m,Not Rated,5.6,31000.0,NaN,...,Dave Freeman,"Elke Sommer, Kenneth Williams, Bernard Bresslaw","Parody, Slapstick, Comedy",United Kingdom,"Bad Godesberg Way, Maidenhead, Berkshire, Engl...","The Rank Organisation, Peter Rogers Productions",English,0,0,0
2,2,tt0131526,"So Young, So Lovely, So Vicious...",https://www.imdb.com/title/tt0131526,1975,1h 30m,NaN,4.9,487.0,NaN,...,"Silvio Amadio, Roberto Natale","Gloria Guida, Dagmar Lassander, Fred Robsahm","Drama, Mystery, Thriller",Italy,"Sardinia, Italy",Domiziana Internazionale Cinematografica,Italian,0,0,0
3,3,tt0073335,Mahogany,https://www.imdb.com/title/tt0073335,1975,1h 49m,PG,6.1,23000.0,NaN,...,"John Byrum, Toni Amber","Diana Ross, Billy Dee Williams, Anthony Perkins","Drama, Romance",United States,Marshall Field Co Department Store 111 N Sta...,"Motown Productions, Nikor Productions, Paramou...","English, Italian",0,0,1
4,4,tt0073006,Foreplay,https://www.imdb.com/title/tt0073006,1975,1h 15m,R,4.0,185.0,NaN,...,"Dan Greenburg, Jack Richardson, Bruce Jay Frie...","Irwin Corey, Pat Paulsen, Deborah Loomis","Parody, Satire, Sketch Comedy, Comedy",United States,NaN,SynFrank Enterprises,English,0,0,0


In [6]:
df_complete_movie_info = (
    pd.merge(
        df_boxoffice_dc_marvel,
        df_imdb_filmes,
        left_on=["FILM", "YEAR"],
        right_on=["TITLE", "YEAR"],
        how="outer",
    )
    .sort_values(by="YEAR", ascending=False)
    .drop_duplicates(keep="last")
)

In [7]:
# ==============================================================================
# 6. FILTRAGEM DE FRANQUIAS (MARVEL E DC)
# ==============================================================================

# Mantendo apenas filmes pertencentes às franquias:
#   - Marvel
#   - DC
#
# Isso elimina registros de outros universos cinematográficos presentes no dataset.

df_complete_dc_marvel = df_complete_movie_info[
    (df_complete_movie_info["FRANCHISE"] == "Marvel") | (df_complete_movie_info["FRANCHISE"] == "DC")
]

In [8]:
# ==============================================================================
# 7. REMOÇÃO DE COLUNAS REDUNDANTES E DESNECESSÁRIAS
# ==============================================================================

# Eliminando colunas:
#   - duplicadas após o merge,
#   - auxiliares,
#   - pouco relevantes para análises finais,
#   - ou já representadas por outras métricas equivalentes.

df_complete_dc_marvel = df_complete_dc_marvel.drop(
    columns=[
        "25X_PROD",
        "BOX_OFFICE_GROSS_DOMESTIC_US_AND_CANADA",
        "BOX_OFFICE_GROSS_WORLDWIDE",
        "BUDGET_y",
        "DURATION",
        "LENGTH",
        "MPAA_RATING",
        "TITLE",
        "YEAR",
    ]
).reset_index(drop=True)

In [9]:
df_complete_dc_marvel.sort_values(by="RATING_IMDB", ascending=False).head(2)

,int64_field_0,FILM,US_RELEASE_DATE,BOX_OFFICE_GROSS_OTHER_TERRITORIES,BUDGET_x,MCU,PHASE,DISTRIBUTOR,MINUTES,FRANCHISE,...,WRITER,STAR,GENRE,COUNTRY_ORIGIN,FILMING_LOCATION,PRODUCTION_COMPANY,LANGUAGE,WIN,NOMINATION,OSCAR
75,36,The Dark Knight,2008-07-18,472349861,185000000,False,NaN,Warner Bros.,152.0,DC,...,"Jonathan Nolan, Christopher Nolan, David S Goyer","Christian Bale, Heath Ledger, Aaron Eckhart","Action Epic, Epic, Superhero, Action, Crime, D...","United States, United Kingdom","Chicago, Illinois, USA","Warner Bros, Legendary Entertainment, Syncopy","English, Mandarin",0,164,0
6,107,Spider-Man: Across the Spider-Verse,2023-06-02,309205354,100000000,False,NaN,Columbia Pictures,140.0,Marvel,...,"Phil Lord, Christopher Miller, Dave Callaham","Shameik Moore, Hailee Steinfeld, Brian Tyree H...","Computer Animation, Superhero, Supernatural Fa...",United States,NaN,"Columbia Pictures, Marvel Entertainment, Avi A...",English,0,164,1


In [10]:
df_tratado = df_complete_dc_marvel.copy()

In [11]:
money_cols = [
    "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
    "BUDGET_x",
    "INFLATION_ADJUSTED_BUDGET",
    "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
    "GROSS_OPENING_WEEKEND",
    "GROSS_US_CANADA",
    "GROSS_WORLD_WIDE",
]

for col in money_cols:
    df_tratado[col] = (
        df_tratado[col]
        .astype(str)
        .str.replace("$", "")
        .str.replace(",", "")
        .replace("None", 0)
        .astype(float)
    )

In [12]:
percent_cols = ["DOMESTIC"]

for col in percent_cols:
    df_tratado[col] = df_tratado[col].astype(str).str.replace("%", "").astype(float)

In [13]:
date_cols = ["US_RELEASE_DATE"]

for col in date_cols:
    df_tratado[col] = pd.to_datetime(df_tratado[col], format="%m/%d/%Y")

In [14]:
int_cols = [
    "VOTE",
    "WIN",
    "OSCAR",
    "NOMINATION",
    "ROTTEN_TOMATOES_CRITIC_SCORE",
]

for col in int_cols:
    df_tratado[col] = df_tratado[col].fillna(0).astype(int)

In [15]:
df_tratado["YEAR"] = df_tratado["US_RELEASE_DATE"].dt.year

In [16]:
df_tratado["GENRE"].str.split(",").str[0].str.strip()

df_tratado["COUNTRY_ORIGIN"] = df_tratado["COUNTRY_ORIGIN"].str.split(",").str[0].str.strip()

df_tratado["LANGUAGE"] = df_tratado["LANGUAGE"].str.split(",").str[0].str.strip()

df_tratado["PRODUCTION_COMPANY"] = df_tratado["PRODUCTION_COMPANY"].str.split(",").str[0].str.strip()

df_tratado["WRITER"] = df_tratado["WRITER"].str.split(",").str[0].str.strip()

In [17]:
df_tratado = df_tratado[sorted(df_tratado.columns)].drop_duplicates()

In [18]:
pandas_gbq.to_gbq(
   df_tratado,
   project_id=meu_gcp_project_id,
   destination_table="silver.movies_dc_marvel",
   if_exists="replace",
   credentials=gcp_credentials, 
)